In [21]:
# ============================================
# APPLICATION
# Détection de Faux Billets en Euros
# ============================================

import pandas as pd
import numpy as np
import pickle
import warnings
warnings.filterwarnings('ignore')

# ============================================
# 1. CHARGEMENT DU MODÈLE SAUVEGARDÉ
# ============================================

# Charger le modèle entraîné
with open('log_reg.pkl', 'rb') as f:
    log_reg = pickle.load(f)

# Charger le scaler entraîné
with open('scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

print(" Modèle chargé avec succès !")

# ============================================
# 2. FONCTION DE PREDICTION - 1 BILLET
# ============================================

def predire_billet(diagonal, height_left,
                   height_right, margin_low,
                   margin_up, length):

    # Création du DataFrame
    billet = pd.DataFrame({
        'diagonal'     : [diagonal],
        'height_left'  : [height_left],
        'height_right' : [height_right],
        'margin_low'   : [margin_low],
        'margin_up'    : [margin_up],
        'length'       : [length]
    })

    # Normalisation
    billet_scaled = scaler.transform(billet)

    # Prédiction
    prediction  = log_reg.predict(
                    billet_scaled)[0]
    probabilite = log_reg.predict_proba(
                    billet_scaled)[0]

    # Résultat
    print("\n" + "=" * 50)
    print(" RÉSULTAT DE L'ANALYSE :")
    print("=" * 50)

    if prediction == True:
        nature    = "VRAI BILLET"
        confiance = probabilite[1] * 100
    else:
        nature    = "FAUX BILLET"
        confiance = probabilite[0] * 100

    print(f"→ Nature     : {nature}")
    print(f"→ Proba Vrai : "
          f"{probabilite[1]*100:.2f}%")
    print(f"→ Proba Faux : "
          f"{probabilite[0]*100:.2f}%")
    print(f"→ Confiance  : {confiance:.2f}%")

    if confiance < 90:
        print(f"\n ATTENTION :")
        print(f"→ Confiance faible !")
        print(f"→ Vérification humaine "
              f"recommandée !")

    print("=" * 50)

    return nature, confiance

# ============================================
# 3. FONCTION DE PREDICTION - PLUSIEURS
# ============================================

def predire_plusieurs_billets(fichier_csv):

    # Chargement
    nouveaux_billets = pd.read_csv(
        fichier_csv, sep=","
    )
    print(f"\n {len(nouveaux_billets)} "
          f"billets chargés !")

    # Récupérer les ids 
    ids = nouveaux_billets.iloc[:, 0]

    # Sélectionner uniquement
    # les bonnes colonnes 
    colonnes = [
        'diagonal',
        'height_left',
        'height_right',
        'margin_low',
        'margin_up',
        'length'
    ]
    nouveaux_billets = nouveaux_billets[colonnes]

    # Normalisation
    billets_scaled = scaler.transform(
        nouveaux_billets
    )

    # Prédictions
    predictions  = log_reg.predict(billets_scaled)
    probabilites = log_reg.predict_proba(
                    billets_scaled)

    # Résultats
    resultats = pd.DataFrame({
        'Billet'     : ids.values,
        'Nature'     : ['VRAI' if p
                        else 'FAUX'
                        for p in predictions],
        'Proba Vrai' : (probabilites[:,1]*100
                       ).round(2),
        'Proba Faux' : (probabilites[:,0]*100
                       ).round(2),
        'Confiance'  : [max(p)*100
                        for p in probabilites]
    })

    resultats['Alerte'] = resultats[
        'Confiance'
    ].apply(
        lambda x: 'Vérifier !'
        if x < 90 else 'OK'
    )

    print("\n" + "=" * 70)
    print("RÉSULTATS DE L'ANALYSE :")
    print("=" * 70)
    print(resultats.to_string(index=False))

    nb_vrais = int(predictions.sum())
    nb_faux  = len(predictions) - nb_vrais

    print("\n" + "=" * 70)
    print("RÉSUMÉ :")
    print("=" * 70)
    print(f"→ Total analysés : "
          f"{len(predictions)}")
    print(f"→ Vrais billets  : {nb_vrais}")
    print(f"→ Faux billets   : {nb_faux} ")

    return resultats

# ============================================
# 4. DEMONSTRATION
# ============================================

print("\n" + "=" * 50)
print(" APPLICATION ONCFM")
print("   Détection de Faux Billets")
print("=" * 50)

# Test 1 : Vrai billet (ligne 803)
print("\n TEST 1 : Vrai billet")
_ = predire_billet(
    diagonal     = 171.87,
    height_left  = 103.27,
    height_right = 104.50,
    margin_low   = 3.85,
    margin_up    = 3.03,
    length       = 113.19
)

# Test 2 : Faux billet (ligne 1306)
print("\n TEST 2 : Faux billet")
_ = predire_billet(
    diagonal     = 172.13,
    height_left  = 103.99,
    height_right = 104.06,
    margin_low   = 5.80,
    margin_up    = 3.28,
    length       = 111.30
)

# ============================================
# 5. TEST FICHIER EVALUATEUR - 5 BILLETS 
# ============================================

print("\n TEST FICHIER EVALUATEUR")
resultats = predire_plusieurs_billets(
    r"C:\Users\QRSY8624\OneDrive - orange.com\Documents\CFA orange\Document_Alternant\Projet12\P12 billets_test.csv"
)


 Modèle chargé avec succès !

 APPLICATION ONCFM
   Détection de Faux Billets

 TEST 1 : Vrai billet

 RÉSULTAT DE L'ANALYSE :
→ Nature     : VRAI BILLET
→ Proba Vrai : 99.91%
→ Proba Faux : 0.09%
→ Confiance  : 99.91%

 TEST 2 : Faux billet

 RÉSULTAT DE L'ANALYSE :
→ Nature     : FAUX BILLET
→ Proba Vrai : 0.00%
→ Proba Faux : 100.00%
→ Confiance  : 100.00%

 TEST FICHIER EVALUATEUR

 5 billets chargés !

RÉSULTATS DE L'ANALYSE :
 Billet Nature  Proba Vrai  Proba Faux  Confiance Alerte
 172.09   VRAI       99.53        0.47  99.530929     OK
 171.52   FAUX        0.34       99.66  99.662834     OK
 171.78   VRAI       99.94        0.06  99.939995     OK
 172.02   FAUX        0.00      100.00  99.997075     OK
 171.79   FAUX        1.14       98.86  98.863869     OK

RÉSUMÉ :
→ Total analysés : 5
→ Vrais billets  : 2
→ Faux billets   : 3 
